프로그램이 커짐에 따라 명확성, 테스트 가능성, 효율성을 유지하면서 증가하는 복잡도를 만족시키는 것은 어려운 일이다. 그 중에서도 단일 스레드 프로그램을 동시 실행되는 여러 흐름으로 이뤄진 프로그램으로 바꾸는 경우이다. 이런 문제를 [콘웨이의 생명 게임](https://ko.wikipedia.org/wiki/%EB%9D%BC%EC%9D%B4%ED%94%84_%EA%B2%8C%EC%9E%84) `Conway Game of Life`를 통해 구현해 보자. 

In [ ]:
ALIVE = '*'
EMPTY = '-'

class Grid:
    def __init__(self, height, width):
        self.height = height
        self.width = width
        self.rows = []
        for _ in range(self.height):
            self.rows.append([EMPTY] * self.width)
    
    def get(self, y, x):
        return self.rows[y % self.height][x % self.width]

    def set(self, y, x, state):
        self.rows[y % self.height][x % self.width] = state

    def __str__(self):
        output = ''
        for row in self.rows:
            for cell in row:
                output += cell
            output += '\n'
        return output

클래스가 작동하는 모습을 보기 위해 Grid 인스턴스를 만들고 초기 상태를 고전적인 글라이더 `틱이 지남에 따라 추기 위치에서 미끄러지듯 한쪽 방향으로 이동하는 모습` 형태로 설정해서 출력해 본다.

In [ ]:
grid = Grid(5, 9)
grid.set(0, 3, ALIVE)
grid.set(1, 4, ALIVE)
grid.set(2, 2, ALIVE)
grid.set(2, 3, ALIVE)
grid.set(2, 4, ALIVE)
print(grid)

콘웨이 생명 게임의 세 가지 규칙을 반영한 게임 로직을 구현해 본다.
1. 이웃한 셀 중에서 두 개 이하가 살아 있으면 가운데 셀이 죽는다.
1. 네 개 이상이 살아 있으면 가운데 셀이 죽는다.
1. 정확히 세 개가 살아 있을 때 가운데 셀이 살아 있으면 계속 살아남고, 빈 셀이 살아 있는 상태로 바뀐다.

In [ ]:
def count_neighbors(y, x, get):
    n_ = get(y - 1, x + 0) # 북(N)
    ne = get(y - 1, x + 1) # 북동(NE)
    e_ = get(y + 0, x + 1) # 동(E)
    se = get(y + 1, x + 1) # 남동(SE)
    s_ = get(y + 1, x + 0) # 남(S)
    sw = get(y + 1, x - 1) # 남서(SW)
    w_ = get(y + 0, x - 1) # 서(W)
    nw = get(y - 1, x - 1) # 북서(NW)
    neighbor_states = [n_, ne, e_, se, s_, sw, w_, nw]
    count = 0
    for state in neighbor_states:
        if state == ALIVE:
            count += 1
    return count

def game_logic(state, neighbors):
    if state == ALIVE:
        if neighbors < 2:
            return EMPTY # 살아 있는 이웃이 너무 적음: 죽음
        elif neighbors > 3:
            return EMPTY # 살아 있는 이웃이 너무 많음: 죽음
    else:
        if neighbors == 3:
            return ALIVE # 다시 생성됨
    return state

def step_cell(y, x, get, set):
    state = get(y, x)
    neighbors = count_neighbors(y, x, get)
    next_state = game_logic(state, neighbors)
    set(y, x, next_state)

def simulate(grid):
    next_grid = Grid(grid.height, grid.width)
    for y in range(grid.height):
        for x in range(grid.width):
            step_cell(y, x, grid.get, next_grid.set)
    return next_grid

grid = simulate(grid)
print(grid)

한 번에 전체 변경사항을 그리드 형태로 출력할수 있는 헬퍼 클래스를 만들어서 출력해 본다.

In [ ]:
class ColumnPrinter:
    def __init__(self):
        self.columns = []

    def append(self, data):
        self.columns.append(data)

    def __str__(self):
        row_count = 1
        for data in self.columns:
            row_count = max(
                row_count, len(data.splitlines()) + 1)

        rows = [''] * row_count
        for j in range(row_count):
            for i, data in enumerate(self.columns):
                line = data.splitlines()[max(0, j - 1)]
                if j == 0:
                    padding = ' ' * (len(line) // 2)
                    rows[j] += padding + str(i) + padding
                else:
                    rows[j] += line

                if (i + 1) < len(self.columns):
                    rows[j] += ' | '

        return '\n'.join(rows)

columns = ColumnPrinter()
for i in range(10):
    columns.append(str(grid))
    grid = simulate(grid)

print(columns)

단일 코어에서 한 스레드로 실행되는 경우에는 현재까지의 코드가 잘 동작한다. 그런데 인터넷을 통한 사용자 통신으로 인해 상태 전이가 결정되는 경우에는 어떨까? 가장 단순한 방법은 블로킹 I/O를 `game_logic` 안에 직접 추가하는 것이다.
```python
def game_logic(state, neighbors):
    # 블로킹 I/O를 여기서 수행한다
    data = my_socket.recv(100)
```

예상하듯 이 접근 방법은 블로킹 I/O 및 그리드 확장으로 인한 성능 저하를 예상할 수 밖에 없다. 해결책은 I/O를 병렬로 수행하는 것이다.
- 팬아웃(fan-out): 각 작업 단위(여기서는 셀) 동시 실행되는 여러 실행 흐름을 만들어 내는 과정
- 팬인(fan-in): 전체 조율하는 프로세스(여기서는 세대) 안에서 다음 단계로 진행하기 전에 동시 작업 단위의 작업이 모두 끝날 때까지 기다리는 과정

파이썬은 팬아웃, 팬인을 지원하는 여러 가지 내장 도구를 제공한다. 남은 장을 통해 각 접근 방식의 장단점을 이해하도록 한다.